# Phase 2: Fair Comparison — Early Fusion 1D CNN vs. Lead-Wise Attention Fusion

This notebook runs the complete experimental benchmark on **PTB-XL** (100 Hz, 5 diagnostic superclasses):
1. **Baseline Model**: Early Fusion 1D CNN (combines all 12 leads at input layer).
2. **Fusion Variant**: Lead-Wise Attention 1D CNN (extracts per-lead features with shared backbone, fuses via learnable attention).
3. **Fair Comparison Protocol**:
   - Controlled optimizer (`AdamW`), learning rate (`1e-3`), batch size (`64`), and patient-aware splits (`folds 1-8` train, `9` val, `10` test).
   - **3 Random Seeds** (`[42, 123, 999]`) per model reporting $\text{mean} \pm \text{std}$ test Macro AUROC.
4. **Deliverables**:
   - Structured results table: Model, Parameters, Macro AUROC, and Per-Class AUROCs.
   - **Learned Lead Attention Weights Plot**: Visualizing the model's clinical lead importance across the test set.

### Step 1: Install Dependencies & Setup Device

In [ ]:
!pip install -q wfdb torch pandas numpy scikit-learn matplotlib tqdm awscli

import ast
import os
import random
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import wfdb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")
if device.type == "cuda":
    print("GPU Model:", torch.cuda.get_device_name(0))

SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]
STANDARD_LEAD_NAMES = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]

### Step 2: Load & Standardize Dataset

In [ ]:
DATA_DIR = Path("data/ptbxl")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Fast sync if data not already present in Colab
if not (DATA_DIR / "ptbxl_database.csv").exists():
    print("Syncing 100 Hz dataset from PhysioNet Open S3 bucket...")
    !aws s3 sync --no-sign-request --exclude "records500/*" s3://physionet-open/ptb-xl/1.0.3/ data/ptbxl/

# 1. Parse Metadata
df = pd.read_csv(DATA_DIR / "ptbxl_database.csv", index_col="ecg_id")
df.scp_codes = df.scp_codes.apply(ast.literal_eval)

scp_df = pd.read_csv(DATA_DIR / "scp_statements.csv", index_col=0)
code_to_class = scp_df[scp_df.diagnostic == 1].diagnostic_class.to_dict()

def map_to_superclasses(scp_dict):
    classes = set()
    for code in scp_dict.keys():
        if code in code_to_class and code_to_class[code] in SUPERCLASSES:
            classes.add(code_to_class[code])
    return list(classes)

df["diagnostic_superclass"] = df.scp_codes.apply(map_to_superclasses)
for sc in SUPERCLASSES:
    df[sc] = df["diagnostic_superclass"].apply(lambda c: int(sc in c))

# 2. Load Signals (100 Hz)
print("Loading 100 Hz signals...")
raw_signals = []
for rel_path in tqdm(df["filename_lr"], desc="Reading signals"):
    sig, _ = wfdb.rdsamp(str(DATA_DIR / rel_path))
    raw_signals.append(sig)
X = np.transpose(np.array(raw_signals, dtype=np.float32), (0, 2, 1))  # (N, 12, 1000)
y = df[SUPERCLASSES].values.astype(np.float32)

# 3. Standardize per lead strictly on training folds (strat_fold <= 8)
train_mask = (df.strat_fold <= 8).values
val_mask = (df.strat_fold == 9).values
test_mask = (df.strat_fold == 10).values

lead_mean = np.mean(X[train_mask], axis=(0, 2), keepdims=True)
lead_std = np.std(X[train_mask], axis=(0, 2), keepdims=True)
lead_std = np.where(lead_std == 0, 1.0, lead_std)
X_norm = (X - lead_mean) / lead_std

print(f"Preprocessed {len(X_norm)} records. Shapes -> X: {X_norm.shape}, y: {y.shape}")

### Step 3: Dataset Wrapper & Seed Generator

In [ ]:
class PTBXLDataset(Dataset):
    def __init__(self, signals: np.ndarray, labels: np.ndarray):
        self.signals = torch.from_numpy(signals).float()
        self.labels = torch.from_numpy(labels).float()
    def __len__(self):
        return len(self.signals)
    def __getitem__(self, idx):
        return self.signals[idx], self.labels[idx]

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# Fixed Validation and Test loaders
val_ds = PTBXLDataset(X_norm[val_mask], y[val_mask])
test_ds = PTBXLDataset(X_norm[test_mask], y[test_mask])
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

def get_train_loader(seed: int = 42) -> DataLoader:
    set_seed(seed)
    train_ds = PTBXLDataset(X_norm[train_mask], y[train_mask])
    return DataLoader(train_ds, batch_size=64, shuffle=True)

### Step 4: Model Architectures

In [ ]:
class ConvBlock1D(nn.Module):
    def __init__(self, in_c, out_c, kernel_size=7, stride=1, padding=3, pool_size=2, dropout=0.1):
        super().__init__()
        self.conv = nn.Conv1d(in_c, out_c, kernel_size=kernel_size, stride=stride, padding=padding, bias=False)
        self.bn = nn.BatchNorm1d(out_c)
        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool1d(kernel_size=pool_size) if pool_size > 1 else nn.Identity()
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        return self.drop(self.pool(self.relu(self.bn(self.conv(x)))))

class EarlyFusionCNN(nn.Module):
    """Baseline Model: early fusion across all 12 leads at input layer"""
    def __init__(self, in_channels=12, num_classes=5, base_filters=32, dropout=0.2):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock1D(in_channels, base_filters, kernel_size=7, padding=3, pool_size=2, dropout=dropout),
            ConvBlock1D(base_filters, base_filters * 2, kernel_size=7, padding=3, pool_size=2, dropout=dropout),
            ConvBlock1D(base_filters * 2, base_filters * 4, kernel_size=5, padding=2, pool_size=2, dropout=dropout),
            ConvBlock1D(base_filters * 4, base_filters * 8, kernel_size=5, padding=2, pool_size=2, dropout=dropout),
        )
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(base_filters * 8, num_classes),
        )

    def forward(self, x):
        feats = self.features(x)
        pooled = self.global_pool(feats).squeeze(-1)
        return self.classifier(pooled)

class LeadAttentionModule(nn.Module):
    """Learnable self-attention weighting across 12 leads"""
    def __init__(self, feature_dim=256, hidden_dim=64):
        super().__init__()
        self.attn_net = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1, bias=False),
        )

    def forward(self, lead_features):
        scores = self.attn_net(lead_features)          # (B, 12, 1)
        weights = F.softmax(scores, dim=1)             # (B, 12, 1)
        fused = torch.sum(weights * lead_features, dim=1) # (B, D)
        return fused, weights.squeeze(-1)              # fused: (B, D), weights: (B, 12)

class LeadAttentionCNN(nn.Module):
    """Lead-Fusion Variant: shared 1D backbone per lead + attention fusion"""
    def __init__(self, num_leads=12, num_classes=5, base_filters=32, dropout=0.2):
        super().__init__()
        self.num_leads = num_leads
        self.backbone = nn.Sequential(
            ConvBlock1D(1, base_filters, kernel_size=7, padding=3, pool_size=2, dropout=dropout),
            ConvBlock1D(base_filters, base_filters * 2, kernel_size=7, padding=3, pool_size=2, dropout=dropout),
            ConvBlock1D(base_filters * 2, base_filters * 4, kernel_size=5, padding=2, pool_size=2, dropout=dropout),
            ConvBlock1D(base_filters * 4, base_filters * 8, kernel_size=5, padding=2, pool_size=2, dropout=dropout),
        )
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        feature_dim = base_filters * 8
        self.attention_fusion = LeadAttentionModule(feature_dim=feature_dim, hidden_dim=64)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feature_dim, num_classes),
        )

    def forward(self, x, return_attention=False):
        B, L, T = x.shape
        x_reshaped = x.view(B * L, 1, T)
        feats = self.backbone(x_reshaped)
        pooled = self.global_pool(feats).view(B, L, -1)
        fused, attn_weights = self.attention_fusion(pooled)
        logits = self.classifier(fused)
        if return_attention:
            return logits, attn_weights
        return logits

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print("Baseline Parameters:   ", f"{count_params(EarlyFusionCNN()):,}")
print("Lead-Fusion Parameters:", f"{count_params(LeadAttentionCNN()):,}")

### Step 5: Training & Evaluation Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item() * len(x)
    return running_loss / len(loader.dataset)

@torch.no_grad()
def eval_model(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    trues, preds = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss = criterion(out, y)
        running_loss += loss.item() * len(x)
        trues.append(y.cpu().numpy())
        preds.append(torch.sigmoid(out).cpu().numpy())
    trues = np.vstack(trues)
    preds = np.vstack(preds)
    scores = roc_auc_score(trues, preds, average=None)
    macro = float(np.mean(scores))
    per_class = {cls: float(s) for cls, s in zip(SUPERCLASSES, scores)}
    return running_loss / len(loader.dataset), macro, per_class

def train_and_eval_seed(model, train_loader, val_loader, test_loader, epochs=15, lr=1e-3, device=device):
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2, min_lr=1e-5)

    best_val_macro = -1.0
    best_state = None

    for epoch in range(1, epochs + 1):
        t_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        v_loss, v_macro, _ = eval_model(model, val_loader, criterion, device)
        scheduler.step(v_macro)

        if v_macro > best_val_macro:
            best_val_macro = v_macro
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        print(f"Epoch {epoch:02d}/{epochs:02d} | Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f} | Val Macro AUROC: {v_macro:.4f} (Best: {best_val_macro:.4f})")

    # Load best checkpoint and evaluate on Test set
    model.load_state_dict(best_state)
    _, test_macro, test_per_class = eval_model(model, test_loader, criterion, device)
    return test_macro, test_per_class, model

### Step 6: Multi-Seed Experiment Runner (3 Seeds: `[42, 123, 999]`)

In [ ]:
SEEDS = [42, 123, 999]
EPOCHS = 15

# 1. Benchmark Baseline (Early Fusion)
print("=" * 65)
print("BENCHMARKING MODEL 1: Early Fusion 1D CNN Baseline")
print("=" * 65)

baseline_macros = []
baseline_per_class = {cls: [] for cls in SUPERCLASSES}

for s in SEEDS:
    print(f"\n>>> Running Baseline with Seed {s} <<<")
    set_seed(s)
    model = EarlyFusionCNN()
    t_loader = get_train_loader(seed=s)
    test_macro, per_cls, _ = train_and_eval_seed(model, t_loader, val_loader, test_loader, epochs=EPOCHS)
    baseline_macros.append(test_macro)
    for cls in SUPERCLASSES:
        baseline_per_class[cls].append(per_cls[cls])
    print(f"Seed {s} Test Macro AUROC: {test_macro:.4f}")

# 2. Benchmark Lead-Fusion Variant
print("\n" + "=" * 65)
print("BENCHMARKING MODEL 2: Lead-Wise Attention 1D CNN Variant")
print("=" * 65)

fusion_macros = []
fusion_per_class = {cls: [] for cls in SUPERCLASSES}
best_fusion_model = None
best_f_macro = -1.0

for s in SEEDS:
    print(f"\n>>> Running Lead-Fusion with Seed {s} <<<")
    set_seed(s)
    model = LeadAttentionCNN()
    t_loader = get_train_loader(seed=s)
    test_macro, per_cls, trained_m = train_and_eval_seed(model, t_loader, val_loader, test_loader, epochs=EPOCHS)
    fusion_macros.append(test_macro)
    for cls in SUPERCLASSES:
        fusion_per_class[cls].append(per_cls[cls])
    print(f"Seed {s} Test Macro AUROC: {test_macro:.4f}")
    if test_macro > best_f_macro:
        best_f_macro = test_macro
        best_fusion_model = trained_m

### Step 7: Results Comparison Table

In [ ]:
count_params = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)

results_data = [
    {
        "Model": "Early Fusion 1D CNN (Baseline)",
        "Parameters": f"{count_params(EarlyFusionCNN()):,}",
        "Macro AUROC": f"{np.mean(baseline_macros):.4f} ± {np.std(baseline_macros):.4f}",
        **{cls: f"{np.mean(baseline_per_class[cls]):.4f} ± {np.std(baseline_per_class[cls]):.4f}" for cls in SUPERCLASSES}
    },
    {
        "Model": "Lead-Wise Attention CNN (Variant)",
        "Parameters": f"{count_params(LeadAttentionCNN()):,}",
        "Macro AUROC": f"{np.mean(fusion_macros):.4f} ± {np.std(fusion_macros):.4f}",
        **{cls: f"{np.mean(fusion_per_class[cls]):.4f} ± {np.std(fusion_per_class[cls]):.4f}" for cls in SUPERCLASSES}
    }
]

df_results = pd.DataFrame(results_data)
print("\n" + "=" * 75)
print("FINAL RESULTS ON TEST FOLD (3-Seed Average ± Std)")
print("=" * 75)
df_results

### Step 8: Plot Learned Attention Weights Per Lead

In [ ]:
# Extract attention weights across all test records
best_fusion_model.eval()
all_attns = []

with torch.no_grad():
    for x, _ in test_loader:
        x = x.to(device)
        _, attn = best_fusion_model(x, return_attention=True)
        all_attns.append(attn.cpu().numpy())

all_attns = np.vstack(all_attns)  # (N_test, 12)
mean_lead_attn = np.mean(all_attns, axis=0)
std_lead_attn = np.std(all_attns, axis=0)

fig, ax = plt.subplots(figsize=(10, 5))
x_pos = np.arange(12)
colors = ["#2b5c8f"] * 6 + ["#e26d5c"] * 6  # Limb vs Precordial

bars = ax.bar(x_pos, mean_lead_attn, yerr=std_lead_attn, capsize=4, color=colors, edgecolor="black", alpha=0.85)
uniform_ref = 1.0 / 12.0
ax.axhline(uniform_ref, color="crimson", linestyle="--", linewidth=1.2, label=f"Uniform Baseline ({uniform_ref:.3f})")

ax.set_xticks(x_pos)
ax.set_xticklabels(STANDARD_LEAD_NAMES, fontsize=11, fontweight="bold")
ax.set_ylabel("Attention Weight", fontsize=12)
ax.set_xlabel("12 ECG Leads (Blue: Limb Leads | Orange: Precordial Leads)", fontsize=11)
ax.set_title("Learned Lead Attention Importance on PTB-XL Test Fold", fontsize=13, fontweight="bold", pad=10)
ax.grid(axis="y", linestyle=":", alpha=0.6)
ax.legend(frameon=True)

for b in bars:
    h = b.get_height()
    ax.annotate(f"{h:.3f}", xy=(b.get_x() + b.get_width() / 2, h), xytext=(0, 5), textcoords="offset points", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("lead_attention_weights.png", dpi=200, bbox_inches="tight")
plt.show()